In [ ]:
#%pip install xgboost catboost optuna

In [ ]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report,accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna

In [ ]:
# load data
train_data = pd.read_csv('train.csv')
train_data.head()

In [ ]:
target = "Irrigation_Need"

X = train_data.drop(columns=[target, "id"], errors="ignore")
y = train_data[target]

print(train_data.head())
print()
print("Target distribution:")
print(y.value_counts())
print()
print("Target proportions:")
print(y.value_counts(normalize=True))


In [ ]:
# Train/test split using only Kaggle train.csv
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

gnb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])

gnb_model.fit(X_train, y_train)


In [ ]:
# Predicted probabilities
probs = gnb_model.predict_proba(X_test)
classes = gnb_model.named_steps["model"].classes_

probs_df = pd.DataFrame(probs, columns=[f"prob_{c}" for c in classes])
probs_df.head()


In [ ]:
# Baseline predictions: choose the class with the highest probability
baseline_preds = classes[np.argmax(probs, axis=1)]

print("Baseline classification report:")
print(classification_report(y_test, baseline_preds, zero_division=0))

print("Baseline balanced accuracy:", round(balanced_accuracy_score(y_test, baseline_preds), 4))


In [ ]:
# Focus on the rare class
focus_class = "High"
focus_metric = "f1"

def threshold_predictions(probs, classes, focus_class, threshold):
    focus_index = np.where(classes == focus_class)[0][0]
    preds = np.empty(probs.shape[0], dtype=object)

    assign_focus = probs[:, focus_index] >= threshold
    preds[assign_focus] = focus_class

    if (~assign_focus).any():
        other_probs = probs[~assign_focus].copy()
        other_probs[:, focus_index] = -np.inf
        preds[~assign_focus] = classes[np.argmax(other_probs, axis=1)]

    return preds

threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.05):
    threshold_preds = threshold_predictions(probs, classes, focus_class, threshold)

    threshold_rows.append({
        "threshold": threshold,
        "predicted_high_count": np.sum(threshold_preds == focus_class),
        "high_precision": precision_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "high_recall": recall_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "high_f1": f1_score(y_test, threshold_preds, labels=[focus_class], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_test, threshold_preds)
    })

threshold_results = pd.DataFrame(threshold_rows)

threshold_results.sort_values("high_f1", ascending=False).head(10)


In [ ]:
# Choose the threshold with the best F1 for the High class
best_row = threshold_results.loc[threshold_results["high_f1"].idxmax()]
chosen_threshold = best_row["threshold"]

threshold_preds = threshold_predictions(
    probs,
    classes,
    focus_class,
    chosen_threshold
)

print("Chosen class:", focus_class)
print("Chosen metric:", focus_metric)
print("Chosen threshold:", round(chosen_threshold, 2))
print()

print("Thresholded classification report:")
print(classification_report(y_test, threshold_preds, zero_division=0))

print("Thresholded balanced accuracy:", round(balanced_accuracy_score(y_test, threshold_preds), 4))


In [ ]:
baseline_high_f1 = f1_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

threshold_high_f1 = f1_score(
    y_test,
    threshold_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

baseline_high_recall = recall_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

threshold_high_recall = recall_score(
    y_test,
    threshold_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

summary = pd.DataFrame([
    {
        "prediction_type": "baseline argmax",
        "threshold": "default",
        "high_f1": baseline_high_f1,
        "high_recall": baseline_high_recall,
        "balanced_accuracy": balanced_accuracy_score(y_test, baseline_preds)
    },
    {
        "prediction_type": "thresholded High",
        "threshold": chosen_threshold,
        "high_f1": threshold_high_f1,
        "high_recall": threshold_high_recall,
        "balanced_accuracy": balanced_accuracy_score(y_test, threshold_preds)
    }
])

summary


I selected the `High` irrigation need class because it is the rarest class and is also important from a practical standpoint: missing fields that need high irrigation could be costly. I used F1-score for the `High` class as my main metric because it balances precision and recall.

The baseline Gaussian Naive Bayes model used the default rule of assigning each row to the class with the highest predicted probability. Then I tested several probability thresholds for the `High` class and chose the threshold that gave the best `High` class F1-score on the held-out test split.

After applying the threshold, the `High` class F1-score changed from the baseline value shown in the summary table to the thresholded value shown in the summary table. The main tradeoff was that changing the threshold affected precision and recall: lowering the threshold usually predicts `High` more often, which can improve recall but may reduce precision.

Compared with stronger models such as Random Forest, XGBoost, LightGBM, or CatBoost, Gaussian Naive Bayes is much simpler and faster, but it usually performs worse because it assumes the features are conditionally independent within each class. The boosted/tree-based models can capture nonlinear relationships and interactions among soil, weather, crop, and irrigation variables more effectively.


baseline High F1 = ~0.627

thresholded High F1 = ~0.698 

chosen threshold = High class